In [ ]:
# Com base no passo anterior (cd-hit_HOST.ipynb) vamos navegar até o diretório onde estão os resultados do script 'bedtools_to_fasta.py', que são os arquivos gff transformados em arquivos fasta.

cd /mnt/dados/victor_baca/outputs/cd-hit

# Dentro do diretorio cd-hit ja temos os diretorios isescan e blastn, onde estão os arquivos fasta gerados a partir dos arquivos gff. ja separado por ISEScan e Blastn.

In [ ]:
# Vamos criar um script python para organizar esses arquivos fasta por sorotipo, criando subpastas para cada sorotipo dentro dos diretórios isescan e blastn.
nano organize_fasta_by_serotype.py

#####################################################################################################################################################
#!/usr/bin/env python3

import os
import shutil
from pathlib import Path

# Subdiretórios que devem ser IGNORADOS
PROTECTED_DIRS = {'Bovine', 'Fish', 'Human'}

# Dados de mapeamento genome -> sorotipo (extraído de genomes.txt)
GENOME_SEROTYPE_MAP = {
    'GCF_002812445.1_ASM281244v1': 'III',
    'GCF_002812465.1_ASM281246v1': 'III',
    'GCF_002812505.1_ASM281250v1': 'III',
    'GCF_002812425.1_ASM281242v1': 'III',
    'GCF_000636115.1_ASM63611v1': 'Ib',
    'GCF_001190805.1_ASM119080v1': 'Ia',
    'GCF_003939065.1_ASM393906v1': 'Ia',
    'GCF_041728555.1_SA2BKE': 'III',
    'GCF_013000945.1_ASM1300094v1': 'Ia',
    'GCA_003186145.1_ASM318614v1': 'Ib',
    'GCF_000967445.1_ASM96744v1': 'Ib',
    'GCA_002881395.1_ASM288139v1': 'Ib',
    'GCF_002881775.1_ASM288177v1': 'Ib',
    'GCF_002882835.1_ASM288283v1': 'Ib',
    'GCF_002882745.1_ASM288274v1': 'Ib',
    'GCF_002881695.1_ASM288169v1': 'Ib',
    'GCF_002882645.1_ASM288264v1': 'Ib',
    'GCF_002881615.1_ASM288161v1': 'Ib',
    'GCF_002882555.1_ASM288255v1': 'Ib',
    'GCF_002882465.1_ASM288246v1': 'Ib',
    'GCF_002882375.1_ASM288237v1': 'Ib',
    'GCF_002881595.1_ASM288159v1': 'Ib',
    'GCF_002881575.1_ASM288157v1': 'Ib',
    'GCF_002881555.1_ASM288155v1': 'Ib',
    'GCF_002881535.1_ASM288153v1': 'Ib',
    'GCF_002881515.1_ASM288151v1': 'Ib',
    'GCF_002882275.1_ASM288227v1': 'Ib',
    'GCF_002881495.1_ASM288149v1': 'Ib',
    'GCF_002882205.1_ASM288220v1': 'Ib',
    'GCF_002882125.1_ASM288212v1': 'Ib',
    'GCF_002881475.1_ASM288147v1': 'Ib',
    'GCF_002881455.1_ASM288145v1': 'Ib',
    'GCA_002881435.1_ASM288143v1': 'Ib',
    'GCF_002881415.1_ASM288141v1': 'Ib',
    'GCF_002882035.1_ASM288203v1': 'Ib',
    'GCF_002881935.1_ASM288193v1': 'Ib',
    'GCF_002881375.1_ASM288137v1': 'Ib',
    'GCF_002881355.1_ASM288135v1': 'Ib',
    'GCF_002881335.1_ASM288133v1': 'Ib',
    'GCF_002881315.1_ASM288131v1': 'Ib',
    'GCF_002881865.1_ASM288186v1': 'Ib',
    'GCF_002881295.1_ASM288129v1': 'Ib',
    'GCF_002881255.1_ASM288125v1': 'Ib',
    'GCF_002881215.1_ASM288121v1': 'Ib',
    'GCF_002881195.1_ASM288119v1': 'Ib',
    'GCF_002881235.1_ASM288123v1': 'Ib',
    'GCF_002881175.1_ASM288117v1': 'Ib',
    'GCF_002881155.1_ASM288115v1': 'Ib',
    'GCF_000302475.2_ASM30247v3': 'Ib',
    'SA11-UEL': 'III',
    'SA10-UEL': 'III',
    'SA9-UEL': 'Ib',
    'SA8-UEL': 'Ib',
    'GCF_002197205.1_ASM219720v1': 'III',
    'GCF_001190885.1_ASM119088v1': 'III',
    'GCF_001592425.1_ASM159242v1': 'III',
    'GCF_001592385.1_ASM159238v1': 'III',
    'GCF_006716245.1_ASM671624v1': 'III',
    'GCF_001266635.1_ASM126663v1': 'Ia',
    'GCF_001552035.1_ASM155203v1': 'III',
    'GCF_001026925.1_ASM102692v1': 'V',
    'GCF_000831125.1_ASM83112v1': 'II',
    'GCF_000831145.1_ASM83114v1': 'II',
    'GCF_000831105.1_ASM83110v1': 'V',
    'GCF_025402775.1_ASM2540277v1': 'V',
    'GCF_025402755.1_ASM2540275v1': 'V',
    'GCF_025402735.1_ASM2540273v1': 'V',
    'GCF_021496865.1_ASM2149686v1': 'V',
    'GCF_021496845.1_ASM2149684v1': 'V',
    'GCF_021496825.1_ASM2149682v1': 'V',
    'GCF_021496945.1_ASM2149694v1': 'V',
    'GCF_021496805.1_ASM2149680v1': 'V',
    'GCF_019552345.1_ASM1955234v1': 'V',
    'GCF_030078235.1_ASM3007823v1': 'V',
    'GCF_030078295.1_ASM3007829v1': 'V',
    'GCF_030078315.1_ASM3007831v1': 'V',
    'GCF_030078355.1_ASM3007835v1': 'V',
    'GCF_030078375.1_ASM3007837v1': 'V',
    'GCF_030078255.1_ASM3007825v1': 'Ia',
    'GCF_030078275.1_ASM3007827v1': 'Ia',
    'GCF_030078335.1_ASM3007833v1': 'Ia',
    'GCF_030078395.1_ASM3007839v1': 'Ia',
    'GCF_030078415.1_ASM3007841v1': 'Ia',
    'GCF_003966545.1_ASM396654v1': 'II',
    'GCF_030323365.1_ASM3032336v1': 'IV',
    'GCF_021655535.1_ASM2165553v1': 'III',
    'GCF_002289205.1_ASM228920v1': 'III',
    'GCF_001275545.2_ASM127554v2': 'III',
    'GCF_002197385.1_ASM219738v1': 'III',
    'GCF_002197365.1_ASM219736v1': 'III',
    'GCF_002197325.1_ASM219732v1': 'VI',
    'GCF_002197305.1_ASM219730v1': 'III',
    'GCF_002197285.1_ASM219728v1': 'III',
    'GCF_002197245.1_ASM219724v1': 'III',
    'GCF_002197265.1_ASM219726v1': 'III',
    'GCF_002197425.1_ASM219742v1': 'III',
    'GCA_016454805.1_ASM1645480v1': 'III',
    'GCF_030489725.1_ASM3048972v1': 'III',
    'GCF_002214425.1_ASM221442v1': 'III',
    'GCF_000427035.1_09mas018883': 'Unknown',
    'GCF_900078265.1_SA111': 'Unknown',
    'GCF_947313835.2_23220_1_11_h_nanopore': 'Unknown',
    'GCF_029625275.1_ASM2962527v1': 'Unknown',
    'GCF_009930895.1_ASM993089v1': 'Unknown',
    'GCF_009930915.1_ASM993091v1': 'Unknown',
    'GCF_029625255.1_ASM2962525v1': 'Unknown',
    'GCF_001708205.1_ASM170820v1': 'Ia',
    'GCF_025631115.1_ASM2563111v1': 'Unknown',
    'GCF_025631095.1_ASM2563109v1': 'Unknown',
    'GCF_025631125.1_ASM2563112v1': 'Unknown',
    'GCF_025801965.1_ASM2580196v1': 'Unknown',
    'GCF_027533695.1_ASM2753369v1': 'Unknown',
    'GCF_024424535.1_ASM2442453v1': 'Unknown',
    'GCF_024424555.1_ASM2442455v1': 'Unknown',
    'GCF_024424515.1_ASM2442451v1': 'Unknown',
    'GCF_014895495.1_ASM1489549v1': 'Unknown',
    'GCA_020980705.1_ASM2098070v1': 'Unknown',
    'GCA_020980735.1_ASM2098073v1': 'Unknown',
    'GCA_020980665.1_ASM2098066v1': 'Unknown',
    'GCF_020980675.1_ASM2098067v1': 'Unknown',
    'GCA_020980325.1_ASM2098032v1': 'Unknown',
    'GCA_020980385.1_ASM2098038v1': 'Unknown',
    'GCF_020980305.1_ASM2098030v1': 'Unknown',
    'GCF_020980285.1_ASM2098028v1': 'Unknown',
    'GCF_019794075.1_ASM1979407v1': 'Unknown',
    'GCF_019794115.1_ASM1979411v1': 'Unknown',
    'GCF_019794045.1_ASM1979404v1': 'Unknown',
    'GCF_019794085.1_ASM1979408v1': 'Unknown',
    'GCF_019794035.1_ASM1979403v1': 'Unknown',
    'GCF_019794015.1_ASM1979401v1': 'Unknown',
    'GCF_019793995.1_ASM1979399v1': 'Unknown',
    'GCF_019793945.1_ASM1979394v1': 'Unknown',
    'GCF_019794635.1_ASM1979463v1': 'Unknown',
    'GCF_019794875.1_ASM1979487v1': 'Unknown',
    'GCF_006543535.1_ASM654353v1': 'Unknown',
    'GCF_006543565.1_ASM654356v1': 'Unknown',
    'GCF_006543545.1_ASM654354v1': 'Unknown',
    'GCF_006543525.1_ASM654352v1': 'Unknown',
    'GCF_006543465.1_ASM654346v1': 'Unknown',
    'GCF_006543275.1_ASM654327v1': 'Unknown',
    'GCF_006543375.1_ASM654337v1': 'Unknown',
    'GCF_006543335.1_ASM654333v1': 'Unknown',
    'GCF_006543305.1_ASM654330v1': 'Unknown',
    'GCF_006543265.1_ASM654326v1': 'Unknown',
    'GCF_006543115.1_ASM654311v1': 'Unknown',
    'GCF_006543125.1_ASM654312v1': 'Unknown',
    'GCF_006543135.1_ASM654313v1': 'Unknown',
    'GCF_003605605.1_ASM360560v1': 'Unknown',
    'GCA_020981825.1_ASM2098182v1': 'Unknown',
    'GCA_020981865.1_ASM2098186v1': 'Unknown',
    'GCA_020981795.1_ASM2098179v1': 'Unknown',
    'GCA_020981845.1_ASM2098184v1': 'Unknown',
    'GCA_020981705.1_ASM2098170v1': 'Unknown',
    'GCA_020981685.1_ASM2098168v1': 'Unknown',
    'GCA_020980445.1_ASM2098044v1': 'Unknown',
    'GCA_020980475.1_ASM2098047v1': 'Unknown',
    'GCA_020980425.1_ASM2098042v1': 'Unknown',
    'GCF_020980405.1_ASM2098040v1': 'Unknown',
    'GCA_020980155.1_ASM2098015v1': 'Unknown',
}

def extract_genome_id(filename):
    """
    Extrai o identificador do genome do nome do arquivo.
    Exemplo: 'GCF_002812445.1_ASM281244v1_genomic_isescan.fasta' -> 'GCF_002812445.1_ASM281244v1'
    """
    base = filename.replace('_genomic_isescan.fasta', '').replace('_genomic_blastn.fasta', '')
    base = base.replace('_isescan.fasta', '').replace('_blastn.fasta', '')
    return base


def get_serotype(genome_id):
    """Retorna o sorotipo baseado no genome_id"""
    return GENOME_SEROTYPE_MAP.get(genome_id, 'Unknown')


def sanitize_folder_name(name):
    """Remove caracteres inválidos para nomes de pasta"""
    return name.replace('/', '_').replace('\\', '_').strip()


def process_directory(base_dir, dir_name):
    """
    Processa um diretório (isescan ou blastn), organizando arquivos por sorotipo
    SOMENTE arquivos DIRETOS no diretório (não entra em subpastas)
    FAZ CÓPIAS ao invés de mover
    """
    dir_path = Path(base_dir) / dir_name
    
    if not dir_path.exists():
        print(f"Diretório {dir_path} não encontrado!")
        return
    
    print(f"\nProcessando diretório: {dir_path}")
    print(f"PROTEGIDO: Subdiretórios {PROTECTED_DIRS} serão ignorados")
    
    copied_files = 0
    skipped_files = 0
    protected_files = 0
    
    # USA glob() ao invés de rglob() - NÃO É RECURSIVO
    for fasta_file in dir_path.glob('*.fasta'):
        # Verifica se é arquivo (não diretório)
        if not fasta_file.is_file():
            continue
        
        filename = fasta_file.name
        genome_id = extract_genome_id(filename)
        serotype = get_serotype(genome_id)
        
        # Ignora se o sorotipo for um dos diretórios protegidos
        if serotype in PROTECTED_DIRS:
            print(f"  ⚠ IGNORADO (protegido): {filename} -> {serotype}/")
            protected_files += 1
            continue
        
        # Cria pasta do sorotipo se não existir
        serotype_folder = dir_path / sanitize_folder_name(serotype)
        serotype_folder.mkdir(exist_ok=True)
        
        # Define destino
        destination = serotype_folder / filename
        
        # Verifica se destino já existe
        if destination.exists():
            print(f"  ⚠ JÁ EXISTE: {filename} em {serotype}/")
            skipped_files += 1
            continue
        
        try:
            # FAZ CÓPIA (preserva metadados)
            shutil.copy2(str(fasta_file), str(destination))
            print(f"  ✓ COPIADO: {filename} -> {serotype}/")
            copied_files += 1
        except Exception as e:
            print(f"  ✗ ERRO: {filename} - {e}")
            skipped_files += 1
    
    print(f"\nResumo {dir_name}:")
    print(f"  Arquivos copiados: {copied_files}")
    print(f"  Arquivos protegidos: {protected_files}")
    print(f"  Arquivos ignorados/erro: {skipped_files}")


def main():
    # Diretório base (ajuste conforme necessário)
    base_dir = '/mnt/dados/victor_baca/outputs/cd-hit'
    
    print("=" * 70)
    print("Organizando arquivos FASTA por sorotipo")
    print("MODO SEGURO: Faz cópias e protege Bovine, Fish, Human")
    print("=" * 70)
    
    # Processa ambos os diretórios
    process_directory(base_dir, 'isescan')
    process_directory(base_dir, 'blastn')
    
    print("\n" + "=" * 70)
    print("Organização concluída!")
    print("IMPORTANTE: Arquivos originais foram PRESERVADOS")
    print("=" * 70)


if __name__ == '__main__':
    main()
#####################################################################################################################################################

# CTRL + O => Salvar  

# CTRL + X => Sair

chmod +x organize_fasta_by_serotype.py

python3 organize_fasta_by_serotype.py

# BLASTn

In [ ]:
# Concatenar os arquivos FASTA das familias de TEs por sorotipo #####################################################################################

## Ir para o diretório cd-hit 
cd /mnt/dados/victor_baca/outputs/cd-hit
## Dentro do diretório cd-hit, criar um novo diretório chamado blastn_cdhit_serotype
mkdir blastn_cdhit_serotype

cd /mnt/dados/victor_baca/outputs/cd-hit/blastn/Ia
cat *.fasta > Ia.fasta
mv /mnt/dados/victor_baca/outputs/cd-hit/blastn/Ia/Ia.fasta /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_serotype

## Como o sorotipo Ib não possui nenhum elemento encontrado pelo BLASTn, o script 'bedtools_to_fasta.py' não gerou nenhum arquivo fasta do BLASTn para esse sorotipo. Portanto o script 'organize_fasta_by_serotype.py' também não criou a pasta Ib dentro do diretório blastn.

cd /mnt/dados/victor_baca/outputs/cd-hit/blastn/III
cat *.fasta > III.fasta
mv /mnt/dados/victor_baca/outputs/cd-hit/blastn/III/III.fasta /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_serotype

## Após concatenar os arquivos fasta por sorotipo, vou criar arquivos concatenados com as combinações possíveis entre os sorotipos Ia e III.
### Ir para o diretório blastn_cdhit_serotype, onde estão os arquivos fasta por sorotipo.
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_serotype 
cat Ia.fasta III.fasta > Ia_III.fasta

In [ ]:
# Ir para o diretorio onde estão os arquivos fasta por sorotipo
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_serotype 

# Script  Para filtrar arquivos FASTA por TEs específicos, removendo os overlaps ja analisados
nano filter_fasta_tes.py # Copiar o código abaixo

#####################################################################################################################################################
#!/usr/bin/env python3

import os
import re
from pathlib import Path

# Lista de TEs para filtrar
TES = [
    'ISSag9','ISSag3','ISSag5','ISSag2','IS1381A','ISSag8','IS861','ISSag4','ISStin1','ISSag12','ISSpy1','ISCco2','IS1167','ISSag11','ISSmu1','IS1548','IS1216E','IS1167A','ISSg1','IS1550','ISSsu4','ISSth2','ISSa4','IS256','ISLgar1','IS1252','IS6770','IS1194','IS-LL6','IS981','ISLla2','ISSsu5','ISSdy1','ISSdy2','IS200S','ISPsy14','ISPst2','ISPpu17','ISShes10','ISSth1','ISScr1','ISStso1','ISSsu7','ISSsu6','IS1239','ISSpy2','ISSag10','ISLgar5','ISBun2','TnSsu5','Tn5531','Tn551','Tn1546','Tn6246','Tn7114','Tn2670','TnSs1','Tn6674','Tn4001','Tn5393','Tn501','Tn5041','Tn6214','Tn2'
]

def header_contains_te(header, te_list):
    """
    Verifica se o cabeçalho contém algum dos TEs da lista.
    
    Args:
        header: String do cabeçalho FASTA
        te_list: Lista de nomes de TEs
        
    Returns:
        bool: True se contém algum TE, False caso contrário
    """
    header_upper = header.upper()
    for te in te_list:
        # Busca case-insensitive pelo nome do TE
        if te.upper() in header_upper:
            return True
    return False

def filter_fasta(input_file, output_file, te_list):
    """
    Filtra um arquivo FASTA mantendo apenas sequências com TEs específicos.
    
    Args:
        input_file: Caminho do arquivo FASTA de entrada
        output_file: Caminho do arquivo FASTA de saída
        te_list: Lista de nomes de TEs para filtrar
    """
    kept_sequences = 0
    total_sequences = 0
    
    with open(input_file, 'r') as fin, open(output_file, 'w') as fout:
        current_header = None
        current_sequence = []
        keep_sequence = False
        
        for line in fin:
            line = line.rstrip('\n')
            
            if line.startswith('>'):
                # Processar sequência anterior se existir
                if current_header:
                    total_sequences += 1
                    if keep_sequence:
                        fout.write(current_header + '\n')
                        fout.write('\n'.join(current_sequence) + '\n')
                        kept_sequences += 1
                
                # Nova sequência
                current_header = line
                current_sequence = []
                keep_sequence = header_contains_te(line, te_list)
            else:
                # Linha de sequência
                if line.strip():  # Ignorar linhas vazias
                    current_sequence.append(line)
        
        # Processar última sequência
        if current_header:
            total_sequences += 1
            if keep_sequence:
                fout.write(current_header + '\n')
                fout.write('\n'.join(current_sequence) + '\n')
                kept_sequences += 1
    
    return total_sequences, kept_sequences

def main():
    """Função principal para processar todos os arquivos FASTA."""
    
    # Criar diretório de saída
    output_dir = Path('filtered')
    output_dir.mkdir(exist_ok=True)
    
    # Encontrar todos os arquivos .fasta no diretório atual
    fasta_files = list(Path('.').glob('*.fasta'))
    
    if not fasta_files:
        print("❌ Nenhum arquivo .fasta encontrado no diretório atual")
        return
    
    print(f"📂 Encontrados {len(fasta_files)} arquivos FASTA")
    print(f"🔍 Filtrando por {len(TES)} TEs")
    print("=" * 70)
    
    total_kept = 0
    total_processed = 0
    
    for fasta_file in fasta_files:
        # Definir arquivo de saída
        output_file = output_dir / f"{fasta_file.stem}_filter.fasta"
        
        try:
            # Filtrar arquivo
            total_seqs, kept_seqs = filter_fasta(fasta_file, output_file, TES)
            
            percentage = (kept_seqs / total_seqs * 100) if total_seqs > 0 else 0
            
            print(f"✅ {fasta_file.name}")
            print(f"   Total: {total_seqs} | Mantidas: {kept_seqs} ({percentage:.1f}%)")
            print(f"   Saída: {output_file}")
            print()
            
            total_kept += kept_seqs
            total_processed += total_seqs
            
        except Exception as e:
            print(f"❌ Erro ao processar {fasta_file.name}: {e}")
            print()
    
    print("=" * 70)
    print(f"📊 RESUMO FINAL:")
    print(f"   Sequências processadas: {total_processed}")
    print(f"   Sequências mantidas: {total_kept}")
    if total_processed > 0:
        print(f"   Percentual mantido: {total_kept/total_processed*100:.1f}%")
    print(f"   Arquivos gerados em: {output_dir}/")

if __name__ == "__main__":
    main()
#####################################################################################################################################################

# CTRL + O = Salvar
# CTRL + X = Sair 

chmod +x filter_fasta_tes.py # Tornar o script executável 

python3 filter_fasta_tes.py

In [ ]:
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST/filtered # Diretório onde estão os arquivos FASTA filtrados do BLASTn

# Script para separar os elementos de Transposons (Tns) e Sequências de Inserção (ISs) de arquivos FASTA em diretórios separados
nano separate_TEs.py

#####################################################################################################################################################
#!/usr/bin/env python3

import os
import re
from pathlib import Path

def create_output_directories():
    """Cria as subpastas Tns e ISs se não existirem"""
    Path("Tns").mkdir(exist_ok=True)
    Path("ISs").mkdir(exist_ok=True)
    print("✓ Diretórios criados: Tns/ e ISs/")

def identify_element_type(header):
    """
    Identifica o tipo de elemento baseado no header.
    Retorna: 'Tn', 'IS', ou None
    """
    # Remove o '>' inicial se houver
    header = header.lstrip('>')
    
    # Padrões mais abrangentes para capturar todos os casos
    # Procura por Tn seguido de letras/números (ex: TnSs1, Tn551, TnSsu5)
    # Procura por In seguido de underscore ou números (ex: In_Tn, In4)
    tn_pattern = r'\b(Tn[A-Za-z0-9]+|In[_0-9])'
    
    # Procura por IS seguido de letras/números (ex: ISSag5, ISCco2, IS1563)
    is_pattern = r'\bIS[A-Za-z0-9]'
    
    # Verifica primeiro por IS (mais específico)
    if re.search(is_pattern, header, re.IGNORECASE):
        return 'IS'
    # Depois verifica por Tn ou In
    elif re.search(tn_pattern, header, re.IGNORECASE):
        return 'Tn'
    
    return None

def process_fasta_file(input_file):
    """
    Processa um arquivo FASTA e separa em Tns e ISs
    """
    base_name = os.path.splitext(os.path.basename(input_file))[0]
    
    # Arquivos de saída
    tn_output = os.path.join("Tns", f"{base_name}_Tns.fasta")
    is_output = os.path.join("ISs", f"{base_name}_ISs.fasta")
    
    # Contadores
    tn_count = 0
    is_count = 0
    other_count = 0
    other_headers = []  # Lista para armazenar headers não classificados
    
    # Abre os arquivos de saída
    with open(tn_output, 'w') as tn_file, \
         open(is_output, 'w') as is_file, \
         open(input_file, 'r') as infile:
        
        current_header = None
        current_sequence = []
        element_type = None
        
        for line in infile:
            line = line.rstrip('\n')
            
            # Nova sequência
            if line.startswith('>'):
                # Salva a sequência anterior se houver
                if current_header and current_sequence:
                    sequence = '\n'.join(current_sequence)
                    
                    if element_type == 'Tn':
                        tn_file.write(f"{current_header}\n{sequence}\n")
                        tn_count += 1
                    elif element_type == 'IS':
                        is_file.write(f"{current_header}\n{sequence}\n")
                        is_count += 1
                    else:
                        other_count += 1
                        other_headers.append(current_header)
                
                # Processa novo header
                current_header = line
                current_sequence = []
                element_type = identify_element_type(line)
                
            else:
                # Adiciona linha de sequência
                if line.strip():  # Ignora linhas vazias
                    current_sequence.append(line)
        
        # Salva a última sequência
        if current_header and current_sequence:
            sequence = '\n'.join(current_sequence)
            
            if element_type == 'Tn':
                tn_file.write(f"{current_header}\n{sequence}\n")
                tn_count += 1
            elif element_type == 'IS':
                is_file.write(f"{current_header}\n{sequence}\n")
                is_count += 1
            else:
                other_count += 1
                other_headers.append(current_header)
    
    return tn_count, is_count, other_count, other_headers

def main():
    """Função principal"""
    print("="*60)
    print("Separador de Elementos Tns e ISs")
    print("="*60)
    
    # Cria diretórios de saída
    create_output_directories()
    
    # Lista todos os arquivos .fasta no diretório atual
    fasta_files = [f for f in os.listdir('.') if f.endswith('.fasta') or f.endswith('.fa')]
    
    if not fasta_files:
        print("\n⚠ Nenhum arquivo FASTA encontrado no diretório atual!")
        return
    
    print(f"\n✓ Encontrados {len(fasta_files)} arquivo(s) FASTA\n")
    
    # Processa cada arquivo
    total_tn = 0
    total_is = 0
    total_other = 0
    all_other_headers = []  # Lista para todos os headers não classificados
    
    for fasta_file in sorted(fasta_files):
        print(f"Processando: {fasta_file}")
        tn_count, is_count, other_count, other_headers = process_fasta_file(fasta_file)
        
        print(f"  ├─ Transposons (Tns): {tn_count}")
        print(f"  ├─ Seq. Inserção (ISs): {is_count}")
        print(f"  └─ Outros elementos: {other_count}")
        
        # Se houver outros elementos, mostra os headers
        if other_count > 0:
            print(f"\n  ⚠ Headers não classificados em {fasta_file}:")
            for header in other_headers:
                print(f"     {header}")
            all_other_headers.extend([(fasta_file, header) for header in other_headers])
        
        print()
        
        total_tn += tn_count
        total_is += is_count
        total_other += other_count
    
    # Resumo final
    print("="*60)
    print("RESUMO GERAL")
    print("="*60)
    print(f"Total de Transposons (Tns): {total_tn}")
    print(f"Total de Seq. Inserção (ISs): {total_is}")
    print(f"Total de outros elementos: {total_other}")
    
    # Mostra resumo dos elementos não classificados
    if total_other > 0:
        print("\n" + "="*60)
        print("ELEMENTOS NÃO CLASSIFICADOS - RESUMO")
        print("="*60)
        for fasta_file, header in all_other_headers:
            print(f"[{fasta_file}] {header}")
    
    print("\n✓ Processamento concluído!")
    print(f"  ├─ Arquivos Tns salvos em: ./Tns/")
    print(f"  └─ Arquivos ISs salvos em: ./ISs/")

if __name__ == "__main__":
    main()
#####################################################################################################################################################

# CTRL + O = Salvar
# CTRL + X = Sair 

chmod +x separate_TEs.py
python3 separate_TEs.py

In [ ]:
# Indo para a pasta de onde estão os arquivos FASTA dos elementos Transposons (Tns)
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_serotype/filtered/Tns
mkdir cd-hit-est # Criando a pasta para guardar os arquivos do CD-HIT-EST

# Adicionando o CD-HIT ao PATH temporariamente 
export PATH=$PATH::/mnt/dados/victor_baca/tools/cdhit

# cd-hit-est versão do CD-HIT para sequências nucleotídicas
nohup cd-hit-est -i Ia_filter_Tns.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Ia_cdhit.fasta &

nohup cd-hit-est -i III_filter_Tns.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/III_cdhit.fasta &

nohup cd-hit-est -i Ia_III_filter_Tns.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Ia_III_cdhit.fasta &

In [ ]:
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_serotype/filtered/Tns/cd-hit-est # Diretório onde estão os arquivos FASTA dos elementos Tns do BLASTn sem redundância

# Script para contar elementos de TEs em arquivos FASTA do BLASTn
nano contar_blastn.py

#####################################################################################################################################################
import re
import sys
from collections import Counter

# Verifica se o arquivo foi fornecido como argumento
if len(sys.argv) < 2:
    print("Uso: python contar_tes.py <caminho_do_arquivo.fasta>")
    print("Exemplo: python contar_tes.py /home/user/dados/Human_cdhit.fasta")
    sys.exit(1)

# Obtém o caminho do arquivo do argumento
filename = sys.argv[1]

# Dicionário para contar os TEs
te_counts = Counter()

# Verifica se o arquivo existe e processa
try:
    with open(filename, 'r') as file:
        for line in file:
            # Processa apenas as linhas de cabeçalho (começam com >)
            if line.startswith('>'):
                # Extrai o nome do TE do cabeçalho
                # Formato esperado: >blast_hit_XX ... Name=TE_NAME;...
                match = re.search(r'Name=([^;]+)', line)
                if match:
                    te_name = match.group(1)
                    te_counts[te_name] += 1

except FileNotFoundError:
    print(f"Erro: Arquivo '{filename}' não encontrado!")
    print("Verifique se o caminho está correto.")
    sys.exit(1)
except Exception as e:
    print(f"Erro ao processar o arquivo: {e}")
    sys.exit(1)

# Verifica se algum TE foi encontrado
if not te_counts:
    print("Aviso: Nenhum elemento transponível encontrado no arquivo.")
    print("Verifique se o arquivo está no formato correto.")
    sys.exit(0)

# Exibe os resultados
print("=" * 60)
print("ELEMENTOS TRANSPONÍVEIS ENCONTRADOS")
print("=" * 60)
print(f"\nTotal de TEs únicos: {len(te_counts)}")
print(f"Total de ocorrências: {sum(te_counts.values())}\n")

print("-" * 60)
print(f"{'Nome do TE':<40} {'Quantidade':>10}")
print("-" * 60)

# Ordena por quantidade (decrescente) e depois por nome
for te_name, count in sorted(te_counts.items(), key=lambda x: (-x[1], x[0])):
    print(f"{te_name:<40} {count:>10}")

print("-" * 60)

# Estatísticas adicionais
print("\n" + "=" * 60)
print("ESTATÍSTICAS")
print("=" * 60)
print(f"TE mais frequente: {te_counts.most_common(1)[0][0]} ({te_counts.most_common(1)[0][1]} ocorrências)")
print(f"Média de ocorrências por TE: {sum(te_counts.values()) / len(te_counts):.2f}")
#####################################################################################################################################################

# CTRL + O -> para salvar o arquivo
# CTRL + X -> para sair do editor

chmod +x contar_blastn.py

python contar_blastn.py Ia_III_cdhit.fasta

python contar_blastn.py Ia_cdhit.fasta

In [ ]:
# Como o elemento ISPa12_ISPpu17 ainda não foi filtrado por ter o ISPpu17 (ISPpu17 um elemento desejado, mais não o ISPa12_ISPpu17), vou criar um novo script para remover/filtrar esse elemento ISPa12_ISPpu17 dos meus arquivos fastas dos elementos ISs

cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_serotype/filtered/ISs # Diretório onde estão os arquivos FASTA dos elementos ISs do BLASTn

nano filter_isppa12.py # Copiar o código abaixo

#####################################################################################################################################################
#!/usr/bin/env python3
"""
Script para filtrar sequências FASTA que contêm ISPa12_ISPpu17 no campo Name.
Remove essas sequências de todos os arquivos FASTA no diretório especificado
e salva os resultados em uma nova subpasta.
"""

import os
import re
from pathlib import Path

def parse_fasta(file_path):
    """
    Lê um arquivo FASTA e retorna uma lista de tuplas (header, sequence).
    """
    sequences = []
    current_header = None
    current_seq = []
    
    with open(file_path, 'r') as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('>'):
                # Salva a sequência anterior se existir
                if current_header is not None:
                    sequences.append((current_header, ''.join(current_seq)))
                # Inicia nova sequência
                current_header = line
                current_seq = []
            else:
                current_seq.append(line)
        
        # Salva a última sequência
        if current_header is not None:
            sequences.append((current_header, ''.join(current_seq)))
    
    return sequences

def has_isppa12_isppu17(header):
    """
    Verifica se o header contém ISPa12_ISPpu17 no campo Name.
    """
    # Procura por Name=...ISPa12_ISPpu17...
    match = re.search(r'Name=([^;]+)', header)
    if match:
        name_value = match.group(1)
        if 'ISPa12_ISPpu17' in name_value:
            return True
    return False

def filter_fasta(input_file, output_file):
    """
    Filtra um arquivo FASTA removendo sequências com ISPa12_ISPpu17 no Name.
    Retorna o número de sequências removidas e mantidas.
    """
    sequences = parse_fasta(input_file)
    
    filtered_sequences = []
    removed_count = 0
    
    for header, seq in sequences:
        if has_isppa12_isppu17(header):
            removed_count += 1
        else:
            filtered_sequences.append((header, seq))
    
    # Escreve o arquivo filtrado
    with open(output_file, 'w') as f:
        for header, seq in filtered_sequences:
            f.write(f"{header}\n{seq}\n")
    
    return removed_count, len(filtered_sequences)

def main():
    # Diretório de entrada
    input_dir = Path('/mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_serotype/filtered/ISs')
    
    # Cria o diretório de saída
    output_dir = input_dir / 'filtered_no_ISPa12_ISPpu17'
    output_dir.mkdir(exist_ok=True)
    
    print(f"Diretório de entrada: {input_dir}")
    print(f"Diretório de saída: {output_dir}")
    print("-" * 70)
    
    # Processa todos os arquivos .fasta no diretório
    fasta_files = list(input_dir.glob('*.fasta'))
    
    if not fasta_files:
        print("Nenhum arquivo FASTA encontrado no diretório!")
        return
    
    total_removed = 0
    total_kept = 0
    
    for fasta_file in fasta_files:
        print(f"\nProcessando: {fasta_file.name}")
        
        output_file = output_dir / fasta_file.name
        
        try:
            removed, kept = filter_fasta(fasta_file, output_file)
            total_removed += removed
            total_kept += kept
            
            print(f"  ✓ Sequências removidas: {removed}")
            print(f"  ✓ Sequências mantidas: {kept}")
            print(f"  ✓ Arquivo salvo: {output_file.name}")
            
        except Exception as e:
            print(f"  ✗ Erro ao processar {fasta_file.name}: {e}")
    
    print("\n" + "=" * 70)
    print(f"RESUMO:")
    print(f"  Arquivos processados: {len(fasta_files)}")
    print(f"  Total de sequências removidas: {total_removed}")
    print(f"  Total de sequências mantidas: {total_kept}")
    print(f"  Arquivos salvos em: {output_dir}")
    print("=" * 70)

if __name__ == "__main__":
    main()
#####################################################################################################################################################

# CTRL + O -> para salvar o arquivo
# CTRL + X -> para sair do editor

chmod +x filter_isppa12.py

python3 filter_isppa12.py

In [ ]:
# Indo para a pasta de onde estão os arquivos FASTA dos elementos Sequências de Inserção (ISs)
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_serotype/filtered/ISs/filtered_no_ISPa12_ISPpu17
mkdir cd-hit-est # Criando a pasta para guardar os arquivos do CD-HIT-EST

# Adicionando o CD-HIT ao PATH temporariamente 
export PATH=$PATH::/mnt/dados/victor_baca/tools/cdhit

# cd-hit-est versão do CD-HIT para sequências nucleotídicas
nohup cd-hit-est -i Ia_filter_ISs.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Ia_cdhit.fasta &

nohup cd-hit-est -i III_filter_ISs.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/III_cdhit.fasta &

nohup cd-hit-est -i Ia_III_filter_ISs.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Ia_III_cdhit.fasta &

In [ ]:
# Ir para o diretório onde estão os arquivos FASTA dos elementos ISs do BLASTn sem redundância
cd /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_HOST/filtered/ISs/filtered_no_ISPa12_ISPpu17/cd-hit-est

# Copiando o script contar_blastn.py para a pasta dos ISs, para poder fazer a contagem dos elementos de ISs
cp /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_serotype/filtered/Tns/cd-hit-est/contar_blastn.py /mnt/dados/victor_baca/outputs/cd-hit/blastn_cdhit_serotype/filtered/ISs/filtered_no_ISPa12_ISPpu17/cd-hit-est

# Utilizando o script contar_blastn.py para contar os elementos de ISs nos arquivos FASTA do CD-HIT-EST

python contar_blastn.py Ia_cdhit.fasta

python contar_blastn.py III_cdhit.fasta

python contar_blastn.py Ia_III_cdhit.fasta

# ISEScan

In [ ]:
# Concatenar os arquivos FASTA das familias de TEs por sorotipo #####################################################################################
cd /mnt/dados/victor_baca/outputs/cd-hit
# Criar diretório para armazenar os arquivos concatenados
mkdir isescan_cdhit_serotype 

cd /mnt/dados/victor_baca/outputs/cd-hit/isescan/Ia
cat *.fasta > Ia.fasta
mv /mnt/dados/victor_baca/outputs/cd-hit/isescan/Ia/Ia.fasta /mnt/dados/victor_baca/outputs/cd-hit/isescan_cdhit_serotype

cd /mnt/dados/victor_baca/outputs/cd-hit/isescan/Ib
cat *.fasta > Ib.fasta
mv /mnt/dados/victor_baca/outputs/cd-hit/isescan/Ib/Ib.fasta /mnt/dados/victor_baca/outputs/cd-hit/isescan_cdhit_serotype

cd /mnt/dados/victor_baca/outputs/cd-hit/isescan/III
cat *.fasta > III.fasta
mv /mnt/dados/victor_baca/outputs/cd-hit/isescan/III/III.fasta /mnt/dados/victor_baca/outputs/cd-hit/isescan_cdhit_serotype

# Após concatenar os arquivos fasta por sorotipo, vou criar arquivos concatenados com as combinações possíveis entre os sorotipos Ia, Ib e III.
## Ir para o diretório isescan_cdhit_serotype, onde estão os arquivos fasta por sorotipo.
cd /mnt/dados/victor_baca/outputs/cd-hit/isescan_cdhit_serotype

cat Ia.fasta Ib.fasta > Ia_Ib.fasta
cat Ia.fasta III.fasta > Ia_III.fasta
cat Ib.fasta III.fasta > Ib_III.fasta
cat Ia.fasta Ib.fasta III.fasta > Ia_Ib_III.fasta

In [ ]:
# Indo para a pasta onde estão os arquivos FASTA concatenados das famílias do ISEScan
cd /mnt/dados/victor_baca/outputs/cd-hit/isescan_cdhit_serotype
mkdir cd-hit-est # Criando a pasta para guardar os arquivos do CD-HIT-EST

# Adicionando o CD-HIT ao PATH temporariamente 
export PATH=$PATH::/mnt/dados/victor_baca/tools/cdhit

# cd-hit-est versão do CD-HIT para sequências nucleotídicas
nohup cd-hit-est -i Ia.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Ia_cdhit.fasta &

nohup cd-hit-est -i Ib.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Ib_cdhit.fasta &

nohup cd-hit-est -i III.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/III_cdhit.fasta &

nohup cd-hit-est -i Ia_Ib.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Ia_Ib_cdhit.fasta &

nohup cd-hit-est -i Ia_III.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Ia_III_cdhit.fasta &

nohup cd-hit-est -i Ib_III.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Ib_III_cdhit.fasta &

nohup cd-hit-est -i Ia_Ib_III.fasta -c 0.80 -d 0 -aS 0.80 -aL 0.80 -r 1 -G 0 -g 1 -o cd-hit-est/Ia_Ib_III_cdhit.fasta &

In [ ]:
cd /mnt/dados/victor_baca/outputs/cd-hit/isescan_cdhit_serotype/cd-hit-est # Diretório onde estão os arquivos FASTA dos elementos ISs do ISEScan sem redundância 

# Criar o script para contar famílias de Insertion Sequences (IS) em arquivo FASTA
nano contar_familias.py

#####################################################################################################################################################
#!/usr/bin/env python3

import re
from collections import Counter
import sys
import os

def parse_fasta_and_count_families(filename):
    """
    Lê arquivo FASTA e conta as famílias de IS encontradas
    
    Args:
        filename: caminho do arquivo FASTA
    
    Returns:
        Counter object com contagem das famílias
    """
    families = []
    
    try:
        with open(filename, 'r') as f:
            for line in f:
                # Processa apenas linhas de cabeçalho (começam com >)
                if line.startswith('>'):
                    # Procura pelo padrão family=NOME_FAMILIA
                    match = re.search(r'family=([^;]+)', line)
                    if match:
                        family_name = match.group(1)
                        families.append(family_name)
    
    except FileNotFoundError:
        print(f"ERRO: Arquivo '{filename}' não encontrado!")
        sys.exit(1)
    except Exception as e:
        print(f"ERRO ao ler arquivo: {e}")
        sys.exit(1)
    
    return Counter(families)

def main():
    # Verifica se o arquivo foi fornecido como argumento
    if len(sys.argv) != 2:
        print("USO: python3 contar_is.py <arquivo.fasta>")
        print("\nExemplo:")
        print("  python3 contar_is.py Bovine_cdhit.fasta")
        print("  python3 contar_is.py /caminho/completo/para/arquivo.fasta")
        sys.exit(1)
    
    filename = sys.argv[1]
    
    # Verifica se o arquivo existe
    if not os.path.isfile(filename):
        print(f"ERRO: Arquivo '{filename}' não encontrado!")
        print(f"Caminho verificado: {os.path.abspath(filename)}")
        sys.exit(1)
    
    print("="*60)
    print("ANÁLISE DE FAMÍLIAS DE INSERTION SEQUENCES (IS)")
    print("="*60)
    print(f"\nArquivo: {filename}")
    print(f"Caminho completo: {os.path.abspath(filename)}\n")
    
    # Faz a contagem
    family_counts = parse_fasta_and_count_families(filename)
    
    if not family_counts:
        print("Nenhuma família de IS encontrada no arquivo!")
        return
    
    # Mostra resultados
    print(f"Total de sequências IS encontradas: {sum(family_counts.values())}")
    print(f"Número de famílias diferentes: {len(family_counts)}\n")
    
    print("-"*60)
    print(f"{'FAMÍLIA':<30} {'QUANTIDADE':>15}")
    print("-"*60)
    
    # Ordena por quantidade (decrescente) e depois por nome
    for family, count in sorted(family_counts.items(), 
                                key=lambda x: (-x[1], x[0])):
        print(f"{family:<30} {count:>15}")
    
    print("-"*60)
    print(f"{'TOTAL':<30} {sum(family_counts.values()):>15}")
    print("="*60)

if __name__ == "__main__":
    main()
#####################################################################################################################################################

chmod +x contar_familias.py

python3 contar_familias.py Ia_cdhit.fasta

python3 contar_familias.py Ib_cdhit.fasta

python3 contar_familias.py III_cdhit.fasta

python3 contar_familias.py Ia_Ib_cdhit.fasta

python3 contar_familias.py Ia_III_cdhit.fasta

python3 contar_familias.py Ib_III_cdhit.fasta

python3 contar_familias.py Ia_Ib_III_cdhit.fasta